# Classical Abel–Jacobi MNIST accuracy results

This notebook runs the same MNIST classification test as `AJ_training_genus30.ipynb` and `scripts/aj_mnist_test_accuracy.py` for:
- **Forward:** classical AJ axis-periodic model (image → AJ map → periodic features → classifier).
- **Inverse:** classical AJ P-function model (image → inverse AJ → classifier).

It then displays the test accuracy and loss in a table and bar chart.

## Config

Set paths for data and optional checkpoints. Use `TEST_SUBSET > 0` for a quick run. Run from repo root or from this `notebooks/` folder.

In [1]:
import os
import re
import subprocess
import sys
from pathlib import Path

# Repo root: works when run from repo root or from notebooks/
REPO_ROOT = Path.cwd()
for p in [Path.cwd(), Path.cwd().parent]:
    if (p / "scripts" / "aj_mnist_test_accuracy.py").exists():
        REPO_ROOT = p
        break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_ROOT = REPO_ROOT / "data"
TABLES_DIR = ""  # e.g. "/path/to/tables" for forward model (aj_integrals_genus30.pt, aj_omegas_genus30.pt)
FORWARD_CKPT = ""  # optional path to forward checkpoint
INVERSE_CKPT = ""  # optional path to inverse checkpoint
TEST_SUBSET = 0    # 0 = full test set; >0 = use first N samples for quick run

## Run accuracy test

Calls `scripts/aj_mnist_test_accuracy.py` with the config above and captures the log.

In [2]:
script = REPO_ROOT / "scripts" / "aj_mnist_test_accuracy.py"
args = [
    sys.executable,
    str(script),
    "--data-root", str(DATA_ROOT),
    "--test-batch-size", "256",
]
if TABLES_DIR:
    args += ["--tables-dir", TABLES_DIR]
if FORWARD_CKPT:
    args += ["--forward-checkpoint", FORWARD_CKPT]
if INVERSE_CKPT:
    args += ["--inverse-checkpoint", INVERSE_CKPT]
if TEST_SUBSET > 0:
    args += ["--test-subset", str(TEST_SUBSET)]

result = subprocess.run(
    args,
    cwd=str(REPO_ROOT),
    capture_output=True,
    text=True,
    timeout=600,
)

log = result.stdout + result.stderr
print(log)
if result.returncode != 0:
    raise RuntimeError(f"Accuracy script exited with code {result.returncode}")

Traceback (most recent call last):
  File "/home/users/hshunt/Abel-Jacobi-Networks/scripts/aj_mnist_test_accuracy.py", line 21, in <module>
    import numpy as np
ModuleNotFoundError: No module named 'numpy'



RuntimeError: Accuracy script exited with code 1

## Parse and show results

Extract the summary lines and display as a table and bar chart.

In [ ]:
def parse_summary(log: str):
    """Parse '  name: accuracy = X.XX%, loss = Y.YYYY' lines after '--- Summary ---'."""
    results = []
    in_summary = False
    for line in log.splitlines():
        if "--- Summary ---" in line:
            in_summary = True
            continue
        if in_summary and line.strip():
            m = re.match(r"\s*(\w+):\s*accuracy\s*=\s*([\d.]+)%\s*,\s*loss\s*=\s*([\d.]+)", line)
            if m:
                results.append({"model": m.group(1), "accuracy": float(m.group(2)), "loss": float(m.group(3))})
        elif in_summary and not line.strip():
            break
    return results

rows = parse_summary(log)
if not rows:
    print("No summary lines found in log.")
else:
    import pandas as pd
    df = pd.DataFrame(rows)
    try:
        display(df)
    except NameError:
        print(df.to_string())

In [ ]:
if rows:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 1, figsize=(6, 3.5))
    names = [r["model"] for r in rows]
    accs = [r["accuracy"] for r in rows]
    ax.bar(names, accs, color=["#2e86ab", "#a23b72"])
    ax.set_ylabel("Test accuracy (%)")
    ax.set_title("Classical Abel–Jacobi MNIST test accuracy")
    ax.set_ylim(0, 105)
    plt.tight_layout()
    plt.show()